# Lab 6: Performance & UI - DAG Inspection and Metadata-Only Filtering - Solution

This notebook contains the complete solution for Lab 6. Use this to verify your implementation or if you get stuck.

## Step 1: Access Spark History Server UI

```bash
# Port-forward to access Spark History Server
kubectl port-forward -n spark svc/spark-history-server 18080:18080

# Or for Docker Compose, it's already available at http://localhost:18080
```

Open your browser to: **http://localhost:18080**

## Step 2: Create Complex Iceberg Tables for Performance Testing

In [ ]:
// Sales fact table with many records
spark.sql("""
  CREATE TABLE iceberg.default.large_sales (
    sale_id INT,
    customer_id INT,
    product_id INT,
    store_id INT,
    sale_date DATE,
    quantity INT,
    unit_price DECIMAL(10,2),
    total_amount DECIMAL(10,2),
    discount DECIMAL(10,2),
    salesperson_id INT
  ) USING iceberg
  PARTITIONED BY (store_id, years(sale_date))
  ORDER BY sale_date, customer_id
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet',
    'write.parquet.compression-codec'='zstd'
  )
""")

In [ ]:
// Generate substantial test data
for (storeId <- 1 to 5) {
  for (year <- 2022 to 2024) {
    for (month <- 1 to 12) {
      val baseDate = s"$year-$month-01"
      for (i <- 1 to 100) {
        spark.sql(s"""
          INSERT INTO iceberg.default.large_sales VALUES
          (${storeId * 10000 + year * 1000 + month * 100 + i},
           ${100 + i % 50},
           ${200 + i % 30},
           $storeId,
           DATE '$baseDate',
           ${1 + i % 10},
           ${(10.0 + i % 100).setScale(2)},
           ${(10.0 + i % 100) * (1 + i % 10).setScale(2)},
           ${(i % 20) * 0.1},
           ${50 + i % 10})
        """)
      }
    }
  }
}

In [ ]:
// Create customer dimension table
spark.sql("""
  CREATE TABLE iceberg.default.large_customers (
    customer_id INT,
    customer_name STRING,
    customer_email STRING,
    customer_segment STRING,
    region STRING,
    signup_date DATE,
    total_purchases INT,
    total_spent DECIMAL(12,2)
  ) USING iceberg
  PARTITIONED BY (region, years(signup_date))
  ORDER BY customer_id
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
  )
""")

In [ ]:
// Generate customer data
for (region <- Seq("west", "east", "north", "south")) {
  for (year <- 2020 to 2023) {
    for (i <- 1 to 50) {
      spark.sql(s"""
        INSERT INTO iceberg.default.large_customers VALUES
        (${i + region.length * 1000},
         'Customer_${region}_${i}',
         'customer${i}@example.com',
         '${List("premium", "standard", "bronze")(i % 3)}',
         '$region',
         DATE '$year-01-01',
         ${10 + i * 5},
         ${(1000.0 + i * 100).setScale(2)})
      """)
    }
  }
}

In [ ]:
// Create product dimension table
spark.sql("""
  CREATE TABLE iceberg.default.large_products (
    product_id INT,
    product_name STRING,
    category STRING,
    subcategory STRING,
    brand STRING,
    unit_price DECIMAL(10,2),
    weight DECIMAL(8,2),
    dimensions STRING
  ) USING iceberg
  PARTITIONED BY (category)
  ORDER BY product_id
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
  )
""")

In [ ]:
// Generate product data
val categories = Seq("Electronics", "Clothing", "Home", "Sports")
for (category <- categories) {
  for (i <- 1 to 30) {
    spark.sql(s"""
      INSERT INTO iceberg.default.large_products VALUES
      (${i + categories.indexOf(category) * 100},
       'Product_${category}_${i}',
       '$category',
       '${category}_Sub${i % 5}',
       'Brand${i % 10}',
       ${(10.0 + i * 5).setScale(2)},
       ${(1.0 + i * 0.1).setScale(2)},
       '${i * 10}x${i * 10}x${i * 10}')
    """)
  }
}

In [ ]:
println("✅ Large tables created successfully!")

## Step 3: Run Complex Iceberg Join Query

In [ ]:
// Complex join query with multiple predicates
val complexQuery = spark.sql("""
  SELECT 
    c.customer_segment,
    p.category,
    s.store_id,
    YEAR(s.sale_date) as sale_year,
    MONTH(s.sale_date) as sale_month,
    SUM(s.total_amount) as total_revenue,
    SUM(s.quantity) as total_quantity,
    AVG(s.unit_price) as avg_unit_price,
    COUNT(DISTINCT c.customer_id) as unique_customers
  FROM iceberg.default.large_sales s
  JOIN iceberg.default.large_customers c ON s.customer_id = c.customer_id
  JOIN iceberg.default.large_products p ON s.product_id = p.product_id
  WHERE s.sale_date >= DATE '2023-01-01'
    AND s.sale_date < DATE '2024-01-01'
    AND c.region IN ('west', 'east')
    AND p.category IN ('Electronics', 'Clothing')
    AND s.total_amount > 50.00
  GROUP BY 
    c.customer_segment,
    p.category,
    s.store_id,
    YEAR(s.sale_date),
    MONTH(s.sale_date)
  ORDER BY total_revenue DESC
""")

In [ ]:
// Time the query execution
val startTime = System.currentTimeMillis()
complexQuery.collect()
val duration = System.currentTimeMillis() - startTime

println(s"Complex join query duration: ${duration}ms")

In [ ]:
// Show results
complexQuery.show()
println("✅ Complex join query executed successfully!")

## Step 5: Analyze Query Plan for Iceberg Optimizations

In [ ]:
// Get detailed query execution plan
val queryPlan = spark.sql("""
  EXPLAIN EXTENDED
  SELECT 
    c.customer_segment,
    p.category,
    s.store_id,
    YEAR(s.sale_date) as sale_year,
    MONTH(s.sale_date) as sale_month,
    SUM(s.total_amount) as total_revenue
  FROM iceberg.default.large_sales s
  JOIN iceberg.default.large_customers c ON s.customer_id = c.customer_id
  JOIN iceberg.default.large_products p ON s.product_id = p.product_id
  WHERE s.sale_date >= DATE '2023-01-01'
    AND s.sale_date < DATE '2024-01-01'
    AND c.region IN ('west', 'east')
    AND p.category IN ('Electronics', 'Clothing')
    AND s.total_amount > 50.00
  GROUP BY 
    c.customer_segment,
    p.category,
    s.store_id,
    YEAR(s.sale_date),
    MONTH(s.sale_date)
""")

In [ ]:
queryPlan.show(truncate=false)
println("✅ Query plan shows Iceberg optimizations!")

## Step 6: Compare Performance With and Without Partition Pruning

In [ ]:
// Query WITHOUT partition predicates (full scan)
val startTime1 = System.currentTimeMillis()
spark.sql("""
  SELECT 
    p.category,
    SUM(s.total_amount) as total_revenue
  FROM iceberg.default.large_sales s
  JOIN iceberg.default.large_products p ON s.product_id = p.product_id
  WHERE s.sale_date >= DATE '2023-01-01'
    AND s.sale_date < DATE '2024-01-01'
  GROUP BY p.category
  ORDER BY total_revenue DESC
""").collect()
val duration1 = System.currentTimeMillis() - startTime1

println(s"Query without partition pruning: ${duration1}ms")

In [ ]:
// Query WITH partition predicates (optimized)
val startTime2 = System.currentTimeMillis()
spark.sql("""
  SELECT 
    p.category,
    SUM(s.total_amount) as total_revenue
  FROM iceberg.default.large_sales s
  JOIN iceberg.default.large_products p ON s.product_id = p.product_id
  WHERE s.sale_date >= DATE '2023-01-01'
    AND s.sale_date < DATE '2024-01-01'
    AND s.store_id = 1  -- Partition predicate
  GROUP BY p.category
  ORDER BY total_revenue DESC
""").collect()
val duration2 = System.currentTimeMillis() - startTime2

println(s"Query with partition pruning: ${duration2}ms")

In [ ]:
// Calculate performance improvement
val improvement = ((duration1 - duration2).toDouble / duration1 * 100)
println(s"Performance improvement: ${improvement}%")
assert(duration2 < duration1, "Partition pruning should improve performance")
println("✅ Partition pruning provides measurable performance improvement!")

## Step 8: Test Metadata-Only Filtering

In [ ]:
// Query that should use metadata-only filtering
spark.sql("""
  EXPLAIN
  SELECT COUNT(*) as record_count
  FROM iceberg.default.large_sales
  WHERE store_id = 1
    AND sale_date >= DATE '2023-01-01'
    AND sale_date < DATE '2024-01-01'
""").show(truncate=false)

In [ ]:
// Run the actual query
val metadataQuery = spark.sql("""
  SELECT COUNT(*) as record_count
  FROM iceberg.default.large_sales
  WHERE store_id = 1
    AND sale_date >= DATE '2023-01-01'
    AND sale_date < DATE '2024-01-01'
""")

In [ ]:
val startTime3 = System.currentTimeMillis()
metadataQuery.collect()
val duration3 = System.currentTimeMillis() - startTime3

println(s"Metadata-only filtered query duration: ${duration3}ms")
println("✅ Metadata-only filtering eliminates unnecessary file scans!")

## ✅ Lab Completion Checklist

- [x] Spark History Server UI accessible on port 18080
- [x] Complex Iceberg join query executed successfully
- [x] Query plan shows Iceberg optimizations
- [x] Partition pruning provides measurable performance improvement
- [x] Metadata-only filtering eliminates unnecessary file scans

## 🎓 Key Concepts Learned

1. **Spark History Server UI**: Web interface for viewing completed Spark jobs
2. **DAG Inspection**: Understanding query execution plans and optimization
3. **Metadata-Only Filtering**: Iceberg's ability to skip files based on metadata
4. **Partition Pruning**: Reducing data scan by eliminating entire partitions
5. **Performance Analysis**: Measuring and comparing query performance
6. **Query Patterns**: How different SQL patterns affect execution plans